# 웨이퍼 맵 불량 패턴 분류 — Colab 학습 노트북 (T4 GPU)

**사전 준비**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 프로젝트 zip(코드 + `data/wm811k_64.npz`)을 구글 드라이브에 업로드하거나 아래 셀에서 직접 업로드

**소요 시간(대략)**: CNN 10~15분, ViT(3 epoch) 40~50분

In [ ]:
# 1) 프로젝트 업로드 및 압축 해제
from google.colab import files
up = files.upload()  # wafer-defect-classification.zip 업로드
!unzip -o -q wafer-defect-classification.zip
%cd wafer-defect-classification
!ls

In [ ]:
# 2) 패키지 설치 (Colab에는 TF/torch 기본 탑재 → transformers만 추가)
!pip -q install transformers

In [ ]:
# 3) (선택) 원본 LSWMD.pkl부터 다시 전처리하려면 실행
#    이미 data/wm811k_64.npz 가 zip에 포함돼 있으면 건너뛰어도 됨
# !python src/prepare_data.py --pkl /content/drive/MyDrive/LSWMD.pkl --out data

In [ ]:
# 4) EDA
!python src/eda.py

In [ ]:
# 5) CNN baseline 학습
%cd src
!python train_cnn.py --epochs 30
%cd ..

In [ ]:
# 6) ViT fine-tuning (T4에서 3 epoch 권장)
%cd src
!python train_vit.py --epochs 3 --batch 32
%cd ..

In [ ]:
# 7) 두 모델 비교 + 오분류 분석 + Grad-CAM
%cd src
!python evaluate.py
!python gradcam.py
%cd ..

from IPython.display import Image, display
for p in ["outputs/figures/model_comparison.png",
          "outputs/figures/confusion_cnn.png",
          "outputs/figures/confusion_vit.png",
          "outputs/figures/misclassified_examples.png",
          "outputs/figures/gradcam_grid.png"]:
    display(Image(p))

In [ ]:
# 8) 결과물 다운로드 (모델 + 그림)
!zip -r -q results.zip models outputs
from google.colab import files
files.download("results.zip")

## 로컬에서 대시보드 실행
```bash
pip install -r requirements.txt
streamlit run app/streamlit_app.py
```
Hugging Face Spaces 배포 시: Space(Streamlit) 생성 → 프로젝트 파일 + `models/` 업로드 → 자동 빌드